In [13]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

In [14]:
# 1. Baca data dari file CSV
df = pd.read_csv("BrentOilPrices.csv")

# 2. Bersihkan nama kolom dari spasi yang tidak terlihat
df.columns = df.columns.str.strip()

# 3. Tentukan nama kolom secara langsung (Manual Fix)
# Kita beri tahu Python bahwa kolom tanggal bernama 'Date' dan harga bernama 'Price'
col_date = "Date" 
col_price = "Price"

# 4. Ubah format kolom tanggal agar bisa dibaca AI
df[col_date] = pd.to_datetime(df[col_date])

# 5. Siapkan tabel baru (df_prophet) dengan nama kolom standar Prophet: 'ds' dan 'y'
df_prophet = df.rename(columns={col_date: 'ds', col_price: 'y'})

# 6. Buang data yang kosong dan urutkan berdasarkan tanggal
df_prophet = df_prophet.sort_values('ds').dropna()

# 7. Tampilkan 5 data teratas untuk memastikan tidak ada error lagi
df_prophet.head()

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_24608\1378488101.py:13: UserWarning:

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.



,ds,y
0,1987-05-20,18.63
1,1987-05-21,18.45
2,1987-05-22,18.55
3,1987-05-25,18.60
4,1987-05-26,18.63


In [15]:
# Pastikan model sudah dilatih dulu
model = Prophet()
model.fit(df_prophet)  # ✅ Gunakan df_prophet yang sudah punya kolom 'ds' dan 'y'

# Buat future dataframe sampai 2032
future_2032 = model.make_future_dataframe(periods=365*7, freq='D')
# 7 tahun ke depan dari data terakhir

# Cetak konfirmasi
print("Model berhasil dilatih dan siap melakukan prediksi!")

print(f"Prediksi dari: {future_2032['ds'].min()}")
print(f"Prediksi sampai: {future_2032['ds'].max()}")
print(f"Total baris: {len(future_2032)}")


00:13:08 - cmdstanpy - INFO - Chain [1] start processing
00:13:10 - cmdstanpy - INFO - Chain [1] done processing


Model berhasil dilatih dan siap melakukan prediksi!
Prediksi dari: 1987-05-20 00:00:00
Prediksi sampai: 2029-11-12 00:00:00
Total baris: 11566


In [16]:
# 1. Melakukan prediksi menggunakan model yang sudah dilatih
# Proses ini akan menghitung nilai prediksi (yhat), batas bawah (yhat_lower), dan batas atas (yhat_upper)
forecast_2032 = model.predict(future_2032)

# 2. Membersihkan urutan data (PENTING: Agar grafik tidak berantakan/garis vertikal)
forecast_2032 = forecast_2032.sort_values('ds').reset_index(drop=True)

# 3. Menampilkan kolom penting dari hasil prediksi
# ds = Tanggal, yhat = Prediksi Harga, yhat_lower/upper = Rentang Ketidakpastian
forecast_2032[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail()

,ds,yhat,yhat_lower,yhat_upper
11561,2029-11-08,121.961941,24.054661,226.819309
11562,2029-11-09,121.878279,15.947328,228.542339
11563,2029-11-10,123.562463,22.361272,227.813157
11564,2029-11-11,123.525572,22.391647,227.944987
11565,2029-11-12,121.776677,22.038318,227.844681


In [17]:
import plotly.graph_objects as go

# 1. Membuat kanvas grafik
fig = go.Figure()

# 2. Menambahkan area "Pita Pink" (Rentang Ketidakpastian)
# yhat_upper adalah batas atas tebakan AI
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat_upper'],
    line=dict(width=0), showlegend=False, hoverinfo="skip"
))

# yhat_lower adalah batas bawah tebakan AI
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat_lower'],
    line=dict(width=0), fill='tonexty', 
    fillcolor='rgba(255, 0, 0, 0.15)', name='Rentang Ketidakpastian'
))

# 3. Menambahkan garis merah (Hasil Prediksi Utama)
fig.add_trace(go.Scatter(
    x=forecast_2032['ds'], y=forecast_2032['yhat'],
    mode='lines', name='Prediksi AI (Prophet)', line=dict(color='red', width=2)
))

# 4. Menambahkan titik-titik hitam (Data Asli/Aktual)
fig.add_trace(go.Scatter(
    x=df_prophet['ds'], y=df_prophet['y'],
    mode='markers', name='Data Aktual', 
    marker=dict(color='black', size=1.5, opacity=0.3)
))

# 5. Mengatur tampilan (Judul dan Nama Sumbu)
fig.update_layout(
    title='Proyeksi Harga Minyak Brent hingga Tahun 2032',
    xaxis_title='Tahun',
    yaxis_title='USD/Barrel',
    hovermode='x unified',
    template='plotly_white'
)

# Tampilkan Grafik
fig.show()